<a href="https://colab.research.google.com/github/aniii052/AML-lab-24cseaiml052/blob/main/AML_exp_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# student = 1,2,3,4,5,6
# x1 = 1,2,3,7,8,9
# x2 = 2,3,4,6,7,8
# x3 = 5,4,3,3,2,1
# class y = 0,0,0,1,1,1
# calculate the person corelation rx1y,rx2y,rx3y,rx1x2,rx2x3, rx1x3 . after that  calculate CFS  marit for candidate subset,
# pipe line = rawdataset -> person correlation -> rcf,rff -> cfs marit -> best subset.


In [8]:
import numpy as np
import pandas as pd
from itertools import combinations

df = pd.DataFrame({
    "student": [1,2,3,4,5,6],
    "x1": [1,2,3,7,8,9],
    "x2": [2,3,4,6,7,8],
    "x3": [5,4,3,3,2,1],
    "y":  [0,0,0,1,1,1]
})

features = ["x1", "x2", "x3"]

corr = df[features + ["y"]].corr()

r_cf = {f: abs(corr.loc[f, "y"]) for f in features}

print("r_cf")
for f, v in r_cf.items():
    print(f"{f}-y = {v:.4f}")

r_ff = {(a,b): abs(corr.loc[a,b])
        for a,b in combinations(features,2)}

print("\nr_ff")
for (a,b), v in r_ff.items():
    print(f"{a}-{b} = {v:.4f}")

def cfs(subset):
    k = len(subset)

    avg_rcf = np.mean([r_cf[f] for f in subset])
    avg_rff = np.mean([r_ff[p] for p in combinations(subset,2)])

    merit = (k * avg_rcf) / np.sqrt(
        k + k*(k-1)*avg_rff
    )

    return avg_rcf, avg_rff, merit

candidates = {
    "S1": ("x1", "x2"),
    "S2": ("x1", "x3"),
    "S3": ("x2", "x3")
}
results = {}

print("\nCFS MERIT")

for name, subset in candidates.items():
    rcf, rff, merit = cfs(subset)
    results[name] = merit
    print(f"{name} = {merit:.4f}")

best = max(results, key=results.get)
print("\nBEST SUBSET")
print(f"{best} = {results[best]:.4f}")

table = []

for name, subset in candidates.items():
    rcf, rff, merit = cfs(subset)

    table.append([name,"{" + ",".join(subset) + "}",rcf,rff,merit])

comparison = pd.DataFrame(table, columns=[
    "Subset", "Selected Feature", "r_cf", "r_ff", "CFS Merit"
])

print("\nCOMPARISON TABLE")
print(comparison.to_string(
    index=False,
    formatters={
        "r_cf": "{:.4f}".format,
        "r_ff": "{:.4f}".format,
        "CFS Merit": "{:.4f}".format
    }
))


r_cf
x1-y = 0.9649
x2-y = 0.9258
x3-y = 0.7746

r_ff
x1-x2 = 0.9926
x1-x3 = 0.9135
x2-x3 = 0.9562

CFS MERIT
S1 = 0.9471
S2 = 0.8892
S3 = 0.8597

BEST SUBSET
S1 = 0.9471

COMPARISON TABLE
Subset Selected Feature   r_cf   r_ff CFS Merit
    S1          {x1,x2} 0.9454 0.9926    0.9471
    S2          {x1,x3} 0.8697 0.9135    0.8892
    S3          {x2,x3} 0.8502 0.9562    0.8597
